# Cinétique chimique : application solution et estimation de paramètres

Ce notebook accompagne les exercices du chapitre 8 sur la cinétique
chimique. Nous calculons d'abord des trajectoires avec
`scipy.integrate.odeint`. Ces trajectoires servent à apprendre l'application
solution discrète

\[
S_h:q\longmapsto \bigl(u(t_j;q)\bigr)_{0\leq j\leq N_t}.
\]

Une fois la machine entraînée, nous la différencions par rapport aux
paramètres cinétiques $q$ pour résoudre un problème inverse, puis nous
préconditionnons ce problème à l'aide des sensibilités.


## Parcours

1. [Réactions et intégration avec `odeint`](#odeint-cinetique)
2. [Données pour l'application solution](#donnees-cinetique)
3. [Une machine qui conserve la masse](#machine-cinetique)
4. [Erreur sur l'application solution](#erreur-cinetique)
5. [Observations et problème inverse](#inverse-cinetique)
6. [Validation par une nouvelle résolution](#validation-cinetique)
7. [Préconditionnement par les sensibilités](#preconditionnement-cinetique)


In [11]:
from time import perf_counter

import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import scipy
from scipy.integrate import odeint

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, SciPy {scipy.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, SciPy 1.18.1


<a id="odeint-cinetique"></a>
## 1. Réactions et intégration avec `odeint`

Nous considérons

\[
A\xrightarrow{q_1}B\xrightarrow{q_2}C,
\qquad
(a(0),b(0),c(0))=(1,0,0),
\]

d'où

\[
a'=-q_1a,\qquad b'=q_1a-q_2b,\qquad c'=q_2b.
\]

La somme $a+b+c$ est conservée. Écrivez le second membre avec la convention `f(t, u, ...)`, puis utilisez l'option `tfirst=True` de [`odeint`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.odeint.html).

In [ ]:
# À compléter.

<a id="donnees-cinetique"></a>
## 2. Données pour l'application solution

Échantillonnez $q$ dans le rectangle $\Theta=[q_{1,\min},q_{1,\max}]	imes[q_{2,\min},q_{2,\max}]$. Pour chaque paramètre, `odeint` produit une sortie de forme `N_TEMPS × 3`. Les données sont donc des couples

\[
(q_i,S_h(q_i))\in\mathbb R^2	imes\mathbb R^{N_t	imes3}.
\]

Séparez apprentissage, validation et test dès la construction.

In [ ]:
# À compléter.

<a id="machine-cinetique"></a>
## 3. Une machine qui conserve la masse

La machine reçoit seulement $q\in\mathbb R^2$ et produit toute la trajectoire discrète. Sa dernière couche calcule $3(N_t-1)$ nombres, les réorganise par instant puis applique `softmax` aux trois concentrations. Ainsi, pour $t_j>0$, les sorties sont positives et leur somme vaut un. La valeur initiale $(1,0,0)$ est ajoutée exactement.

Cette structure n'est pas apprise : elle est imposée par la définition de la machine.

In [ ]:
# À compléter.

In [ ]:
# À compléter.

<a id="erreur-cinetique"></a>
## 4. Erreur sur l'application solution

Calculez les erreurs $L^2$ discrète et $L^\infty$ pour chaque paramètre de test. Vérifiez séparément la positivité, la condition initiale et la conservation. Représentez quelques trajectoires qui n'ont pas servi à l'apprentissage.

In [ ]:
# À compléter.

<a id="inverse-cinetique"></a>
## 5. Observations et problème inverse

Choisissez un paramètre $q_*$ inconnu de l'algorithme et observez seulement
$b$ et $c$ à six instants. Ajoutez de petites perturbations construites. Nous
cherchons ensuite

\[
\widehat q\in\operatorname*{argmin}_{q\in\Theta}
\frac1{|I|}\sum_{j\in I}
\left\|P\Phi(q,p)_j-z_j\right\|^2,
\qquad P(a,b,c)=(b,c).
\]

Pour respecter $q\in\Theta$, posez
$q(r)=q_{\min}+(q_{\max}-q_{\min})\operatorname{sigmoid}(r)$ et optimisez
$r\in\mathbb R^2$.


In [ ]:
# À compléter.

In [8]:
Q_MIN_JAX = jnp.asarray(Q_MIN)
Q_MAX_JAX = jnp.asarray(Q_MAX)
Q_CENTRE_JAX = jnp.asarray(Q_CENTRE)
Q_DEMI_LARGEUR_JAX = jnp.asarray(Q_DEMI_LARGEUR)
INDICES_OBS_JAX = jnp.asarray(indices_observations)
OBSERVATIONS_JAX = jnp.asarray(observations)


def q_depuis_r(r):
    return Q_MIN_JAX + (Q_MAX_JAX - Q_MIN_JAX) * jax.nn.sigmoid(r)


def objectif_inverse(machine, r):
    q = q_depuis_r(r)
    q_n = ((q - Q_CENTRE_JAX) / Q_DEMI_LARGEUR_JAX).reshape(1, 2)
    prediction = machine(q_n)[0, INDICES_OBS_JAX, 1:]
    return jnp.mean((prediction - OBSERVATIONS_JAX) ** 2)


@nnx.jit
def valeur_gradient_inverse(machine, r):
    return jax.value_and_grad(objectif_inverse, argnums=1)(machine, r)

In [ ]:
# À compléter.

<a id="validation-cinetique"></a>
## 6. Validation par une nouvelle résolution

L'optimisation précédente n'a appelé que la machine. Recalculez maintenant **une seule trajectoire** avec `odeint` au paramètre estimé. Comparez :

- les observations ;
- la machine au paramètre estimé ;
- `odeint` au paramètre estimé ;
- `odeint` au paramètre exact, disponible uniquement dans cette expérience construite.

Une petite valeur de $\mathcal J_p$ ne suffit pas : le problème inverse hérite de l'erreur de l'opérateur appris.

In [ ]:
# À compléter.

<a id="preconditionnement-cinetique"></a>
## 7. Préconditionnement par les sensibilités

Nous revenons aux coordonnées physiques $q$. Si

\[
r(q)=\frac{\mathcal O_p(q)-z}{\sqrt{2|I|}},
\qquad
J_p(q)=D_qr(q),
\]

alors $\mathcal J_p(q)=\tfrac12\|r(q)\|^2$ et

\[
\nabla\mathcal J_p(q)=J_p(q)^\top r(q),
\qquad
G_p(q)=J_p(q)^\top J_p(q).
\]

Calculez le Jacobien par différentiation automatique. Comparez le gradient
euclidien avec la direction de Gauss--Newton amortie. Les valeurs propres de
$G_p$ mesurent quelles combinaisons de paramètres sont visibles dans les
observations.


In [ ]:
# À compléter.

In [ ]:
# À compléter.

## Bilan

L'application solution apprise est une machine mathématique dont l'entrée est
le paramètre cinétique et la sortie une trajectoire entière. La structure de
sortie impose positivité, condition initiale et conservation. Le problème
inverse montre ensuite trois effets distincts : approximation de la machine,
observations et optimisation.

Les sensibilités de la machine fournissent enfin une métrique naturelle sur
les paramètres. Dans ce cadre, Gauss--Newton, gradient naturel et
préconditionnement décrivent le même calcul depuis trois points de vue.
